In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import json
from nltk.tokenize import RegexpTokenizer
import numpy as np

1. As per the JD only who are in India and willing to relocate candidates only considered.
2. One more information acutally needed as per the JD i.e. VISA sponsership required for the candidate or not.

However we don't have information about 2 in the dataset, hence used only 1st condition to create dataset

In [ ]:
#Creating data from json file

json_file = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/candidates.jsonl"

profiles = []
career_history = []
education = []
skills = []
certifications = []
languages = []
redrob_signals = []
industries = []
with open(json_file, "r", encoding="utf-8") as f:
    for line in f:
        candidate = json.loads(line)

        candidate_id = candidate["candidate_id"]

        profile = candidate.get("profile", {})
        signal = candidate.get("redrob_signals", {})
      #Filter the candidates who are not in India.
        if((profile.get("country") != "India")):
          continue
      #Filter the candidates who are not willing to relocate (There is VISA sponsership required information hence only used this).
        if (signal.get("willing_to_relocate") == False ):
          continue

        # -------------------------------
        # Candidate Profile
        # -------------------------------
        profiles.append({
            "candidate_id": candidate_id,
            "name": profile.get("anonymized_name"),
            "headline": profile.get("headline"),
            "summary": profile.get("summary"),
            "location": profile.get("location"),
            "country": profile.get("country"),
            "years_of_exp": profile.get("years_of_experience"),
            "current_title": profile.get("current_title"),
            "current_company": profile.get("current_company"),
            "current_company_size": profile.get("current_company_size"),
            "current_industry": profile.get("current_industry")
        })

        # Industry from current profile
        industries.append({
            "candidate_id": candidate_id,
            "Type_of_org": profile.get("current_industry"),
            "company_size": profile.get("current_company_size"),

        })
        candidate_id = candidate.get("candidate_id")
        companies = []
        titles = []
        start_dates = []
        end_dates = []
        industries = []
        company_sizes = []
        experience_descriptions = []

        # -------------------------------
        # Career History
        # -------------------------------
        for exp in candidate.get("career_history", []):
          # Convert months to years
          duration_months = exp.get("duration_months", 0)


          try:
               years = round(float(duration_months) / 12, 1)
          except:
               years = 0

          description = exp.get("description", "")

          enhanced_description = (
             f"{description} Worked here for about {years} years."
          ).strip()

          companies.append(str(exp.get("company", "")))
          titles.append(str(exp.get("title", "")))
          start_dates.append(str(exp.get("start_date", "")))
          end_dates.append(str(exp.get("end_date", "")))
          industries.append(str(exp.get("industry", "")))
          company_sizes.append(str(exp.get("company_size", "")))
          experience_descriptions.append(enhanced_description)

        career_history.append({
            "candidate_id": candidate_id,
            "companies": ", ".join(companies),
            "titles": ", ".join(titles),
            "start_dates": ", ".join(start_dates),
            "end_dates": ", ".join(end_dates),
            "industries": ", ".join(industries),
            "company_sizes": ", ".join(company_sizes),
             "career_history": " | ".join(experience_descriptions)
            })


            # Industry from career history
        industries.append({
                "candidate_id": candidate_id,
                "Type_of_org": exp.get("industry"),
                "company_size": exp.get("company_size"),
            })

        # -------------------------------
        # Education
        # -------------------------------
        for edu in candidate.get("education", []):
            education.append({
                "candidate_id": candidate_id,
                "institution": edu.get("institution"),
                "degree": edu.get("degree"),
                "field_of_study": edu.get("field_of_study"),
                "start_year": edu.get("start_year"),
                "end_year": edu.get("end_year"),
                "grade": edu.get("grade"),
                "tier": edu.get("tier")
            })

        # -------------------------------
        # Skills
        # -------------------------------
        candidate_skills = []
        for skill in candidate.get("skills", []):
          try:
              years = round(float(skill.get("duration_months", 0)) / 12, 1)
          except:
              years = 0

          skill_text = (
            f"{skill.get('name', '')} "
            f"({skill.get('proficiency', 'Unknown')}, "
            f"{skill.get('endorsements', 0)} endorsements, "
            f"{years} years)"
          )

          candidate_skills.append(skill_text)


        skills.append({
          "candidate_id": candidate_id,
          "skills": ", ".join(candidate_skills)
       })

        # -------------------------------
        # Certifications
        # -------------------------------
        for cert in candidate.get("certifications", []):
            certifications.append({
                "candidate_id": candidate_id,
                "name": cert.get("name"),
                "issuer": cert.get("issuer"),
                "year": cert.get("year")
            })

        # -------------------------------
        # Languages
        # -------------------------------
        for lang in candidate.get("languages", []):
            languages.append({
                "candidate_id": candidate_id,
                "language": lang.get("language"),
                "proficiency": lang.get("proficiency")
            })

        # -------------------------------
        # Redrob Signals
        # -------------------------------


        redrob_signals.append({
            "candidate_id": candidate_id,
            "profile_completeness_score": signal.get("profile_completeness_score"),
            "signup_date": signal.get("signup_date"),
            "last_active_date": signal.get("last_active_date"),
            "open_to_work_flag": signal.get("open_to_work_flag"),
            "profile_views_received_30d": signal.get("profile_views_received_30d"),
            "applications_submitted_30d": signal.get("applications_submitted_30d"),
            "recruiter_response_rate": signal.get("recruiter_response_rate"),
            "avg_response_time_hours": signal.get("avg_response_time_hours"),
            "skill_assessment_scores": json.dumps(
                signal.get("skill_assessment_scores", {})
            ),
            "connection_count": signal.get("connection_count"),
            "endorsements_received": signal.get("endorsements_received"),
            "notice_period_days": signal.get("notice_period_days"),
            "expected_min_salary_lpa":
                signal.get("expected_salary_range_inr_lpa", {}).get("min"),
            "expected_max_salary_lpa":
                signal.get("expected_salary_range_inr_lpa", {}).get("max"),
            "preferred_work_mode": signal.get("preferred_work_mode"),
            "willing_to_relocate": signal.get("willing_to_relocate"),
            "github_activity_score": signal.get("github_activity_score"),
            "search_appearance_30d": signal.get("search_appearance_30d"),
            "saved_by_recruiters_30d": signal.get("saved_by_recruiters_30d"),
            "interview_completion_rate": signal.get("interview_completion_rate"),
            "offer_acceptance_rate": signal.get("offer_acceptance_rate"),
            "verified_email": signal.get("verified_email"),
            "verified_phone": signal.get("verified_phone"),
            "linkedin_connected": signal.get("linkedin_connected")
        })






# ==========================================
# Create CSV files
# ==========================================
datasetPath = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/"
pd.DataFrame(profiles).to_csv(
    datasetPath+"candidate_profile.csv", index=False)

pd.DataFrame(career_history).to_csv(
    datasetPath+"candidate_career_history.csv", index=False)

pd.DataFrame(education).to_csv(
    datasetPath+"candidate_education.csv", index=False)

pd.DataFrame(skills).to_csv(
    datasetPath+"candidate_skills.csv", index=False)

pd.DataFrame(certifications).to_csv(
    datasetPath+"candidate_certifications.csv", index=False)

pd.DataFrame(languages).to_csv(
    datasetPath+"candidate_languages.csv", index=False)

pd.DataFrame(redrob_signals).to_csv(
    datasetPath+"candidate_redrob_signals.csv", index=False)

pd.DataFrame(industries).to_csv(
    datasetPath+"candidate_org.csv", index=False)
print("All CSV files generated successfully.")

All CSV files generated successfully.


Creating Vector database for the candidates based on the skills, summary and career history

In [ ]:
CANDIDATE_PROFILE = '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_profile.csv'
CANDIDATE_SKILLS = '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_skills.csv'
CANDIDATE_CAREER_HISTORY = '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_career_history.csv'
CANDIDATE_REDROB_SIGNALS= '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_redrob_signals.csv'

In [4]:
!pip uninstall -y faiss faiss-cpu
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.6 MB/s eta 0:00:00


In [5]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.5.1
    Uninstalling sentence-transformers-5.5.1:
      Successfully uninstalled sentence-transformers-5.5.1


In [6]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
FAISS_INDEX_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/database/candSummaryvector_db.faiss"

METADATA_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/database/candSummarymetadata.pkl"

MODEL_NAME = "all-MiniLM-L6-v2"

In [ ]:



# =====================================================
# LOAD FILES
# =====================================================

profile_df = pd.read_csv(CANDIDATE_PROFILE)

skills_df = pd.read_csv(CANDIDATE_SKILLS)

career_df = pd.read_csv(CANDIDATE_CAREER_HISTORY)

print("Profile Rows :", len(profile_df))
print("Skills Rows  :", len(skills_df))
print("Career Rows  :", len(career_df))

# =====================================================
# KEEP REQUIRED COLUMNS
# =====================================================

skills_df = skills_df[[
    "candidate_id",
    "skills"
]]

career_df = career_df[[
    "candidate_id",
    "companies",
    "titles",
    "industries",
    "career_history"
]]

# =====================================================
# MERGE DATA
# =====================================================

df = profile_df.merge(
    skills_df,
    on="candidate_id",
    how="left"
)

df = df.merge(
    career_df,
    on="candidate_id",
    how="left"
)

# =====================================================
# HANDLE NULLS
# =====================================================

columns_to_fill = [
    "summary",
    "skills",
    "companies",
    "titles",
    "industries",
    "career_history"
]

for col in columns_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna("")

# =====================================================
# CREATE COMBINED TEXT
# =====================================================

df["combined_text"] = (
    "Summary: " + df["summary"].astype(str) +
    "\nSkills: " + df["skills"].astype(str) +
    "\nCompanies: " + df["companies"].astype(str) +
    "\nJob Titles: " + df["titles"].astype(str) +
    "\nIndustries: " + df["industries"].astype(str) +
    "\nCareer History: " + df["career_history"].astype(str)
)

texts = df["combined_text"].tolist()

print("Documents:", len(texts))

# =====================================================
# LOAD MODEL
# =====================================================

print("Loading embedding model...")

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE EMBEDDINGS
# =====================================================

print("Generating embeddings...")

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

print("Embedding Shape:", embeddings.shape)

# =====================================================
# CREATE FAISS INDEX
# =====================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors Stored:", index.ntotal)

# =====================================================
# SAVE INDEX
# =====================================================

faiss.write_index(
    index,
    FAISS_INDEX_FILE
)

print("FAISS index saved")

# =====================================================
# SAVE METADATA
# =====================================================

metadata = df.to_dict("records")

with open(METADATA_FILE, "wb") as f:
    pickle.dump(metadata, f)

print("Metadata saved")
print("Done!")

Profile Rows : 21673
Skills Rows  : 21673
Career Rows  : 21673
Documents: 21673
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/678 [00:00<?, ?it/s]

Embedding Shape: (21673, 384)
Vectors Stored: 21673
FAISS index saved
Metadata saved
Done!


Getting Sematic score of candidates skills, summary and previous work experience with JD

In [ ]:
!pip install -q transformers accelerate torch python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.8 MB/s eta 0:00:00


In [8]:
JSON_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/data/cleaned/job_requirements.json"

In [9]:
import json

In [ ]:
from docx import Document

In [ ]:
# =====================================================
# LOAD JOB REQUIREMENTS JSON
# =====================================================

with open(JSON_FILE, "r") as f:
    jd = json.load(f)

# =====================================================
# BUILD SEMANTIC QUERY
# =====================================================

core_skills = [
    s["skill_id"]
    for s in jd["skills"]["core_required"]
]

secondary_skills = [
    s["skill_id"]
    for s in jd["skills"]["secondary_skills"]
]

domains = [
    d["domain_id"]
    for d in jd["experience_requirements"]["required_domains"]
]

roles = jd["roles"]["preferred_titles"]

industries = jd["company"]["industry_constraints"]["preferred_industries"]

cities = jd["location"]["domestic"]["preferred_cities"]

jd_text = f"""
Hiring for an AI/ML Search Engineer.

Preferred Roles:
{', '.join(roles)}

Mandatory Skills:
{', '.join(core_skills)}

Preferred Skills:
{', '.join(secondary_skills)}

Required Domains:
{', '.join(domains)}

Preferred Industries:
{', '.join(industries)}

Preferred Locations:
{', '.join(cities)}

Experience:
{jd['experience']['min_years']} to {jd['experience']['max_years']} years

Engineering Style:
{jd['work_style']['engineering_style']}

Communication Model:
{jd['work_style']['communication_model']}
"""

print("=" * 100)
print("JD QUERY")
print("=" * 100)
print(jd_text)

# =====================================================
# LOAD EMBEDDING MODEL
# =====================================================

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE JD EMBEDDING
# =====================================================

jd_embedding = model.encode(
    [jd_text],
    normalize_embeddings=True
).astype(np.float32)

JD QUERY

Hiring for an AI/ML Search Engineer.

Preferred Roles:
SENIOR_AI_ENGINEER, ML_ENGINEER, SEARCH_ENGINEER, RANKING_ENGINEER, RECOMMENDER_SYSTEMS_ENGINEER, APPLIED_SCIENTIST

Mandatory Skills:
EMBEDDINGS, RETRIEVAL_SYSTEMS, VECTOR_DB, HYBRID_SEARCH, PYTHON, RANKING_SYSTEMS_EVALUATION, EVALUATION_FRAMEWORKS, AB_TESTING

Preferred Skills:
LLM_FINE_TUNING, LEARNING_TO_RANK, HR_TECH, DISTRIBUTED_SYSTEMS, OPEN_SOURCE_CONTRIBUTION

Required Domains:
RETRIEVAL_SYSTEMS, RANKING_SYSTEMS, SEARCH_SYSTEMS, ML_PRODUCTION_SYSTEMS

Preferred Industries:
AI_NATIVE_TALENT_INTELLIGENCE, RECRUITMENT_TECH, SEARCH_INFRASTRUCTURE, RECOMMENDATION_SYSTEMS, HR_TECH, APPLIED_ML

Preferred Locations:
PUNE, NOIDA

Experience:
5 to 9 years

Engineering Style:
FULL_STACK_ML_SYSTEMS

Communication Model:
ASYNC_WRITING_HEAVY



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# =====================================================
# LOAD FAISS
# =====================================================

index = faiss.read_index(FAISS_INDEX_FILE)

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

print("Candidates:", len(metadata))

# =====================================================
# SEARCH ALL CANDIDATES
# =====================================================

k = index.ntotal

scores, indices = index.search(
    jd_embedding,
    k
)

Candidates: 21673


In [ ]:
# =====================================================
# BUILD RESULT CSV
# =====================================================

results = []

for rank, idx in enumerate(indices[0]):

    similarity = float(scores[0][rank])

    # cosine similarity -> score 0-1
    semantic_score = (similarity + 1) / 2


    candidate = metadata[idx]

    results.append({
        "candidate_id": candidate.get("candidate_id"),
        "semantic_similarity": round(similarity, 4),
        "semantic_score": semantic_score
    })

# =====================================================
# SAVE OUTPUT
# =====================================================

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    "semantic_score",
    ascending=False
)

OUTPUT_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores.csv"

result_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")
print(result_df.head(20))

Saved: /content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores.csv
    candidate_id  semantic_similarity  semantic_score
0   CAND_0098454               0.6655        0.832732
1   CAND_0094728               0.6534        0.826703
2   CAND_0046020               0.6451        0.822554
3   CAND_0072721               0.6434        0.821724
4   CAND_0061175               0.6416        0.820814
5   CAND_0024203               0.6415        0.820757
6   CAND_0087804               0.6398        0.819914
7   CAND_0082618               0.6388        0.819378
8   CAND_0068932               0.6378        0.818909
9   CAND_0016657               0.6371        0.818567
10  CAND_0013314               0.6356        0.817798
11  CAND_0001405               0.6354        0.817720
12  CAND_0043860               0.6352        0.817613
13  CAND_0069638               0.6347        0.817339
14  CAND_0093540               0.6338        0.816879
15  CAND_0051310            

Creating a score for redrob signals. For candidate Notice period, is candidate active or inactive, candidate acceptance rate, candidate prefered work mode(flexible, Hybrid,onsite or Remote), candidate is open to work or not


In [ ]:
df = pd.read_csv(CANDIDATE_REDROB_SIGNALS)

# =====================================================
# LAST ACTIVE DATE SCORE
# More recent activity = higher score
# =====================================================

df['last_active_date'] = pd.to_datetime(
    df['last_active_date'],
    errors='coerce'
)

today = pd.Timestamp.today().normalize()

days_since_active = (
    today - df['last_active_date']
).dt.days

# Cap at 365 days
days_since_active = days_since_active.clip(
    lower=0,
    upper=365
)

df['last_active_score'] = (
    100 - ((days_since_active / 365) * 200)
).round(2)

# Missing dates
df['last_active_score'] = df['last_active_score'].fillna(0)

# =====================================================
# OPEN TO WORK SCORE
# =====================================================

df['open_to_work_score'] = np.where(
    df['open_to_work_flag']
      .astype(str)
      .str.lower()
      .isin(['true', 'yes', '1', 'y']),
    100,
    -100
)

# =====================================================
# NOTICE PERIOD SCORE
# Less notice period = higher score
# till 60days considering it as 100% because 30 days buy out option is there
# =====================================================

def calculate_notice_score(days):

    if pd.isna(days):
        return 0

    if days <= 60:
        return 100

    elif days <= 90:
        return 50

    elif days <= 120:
        return -50

    else:
        return -100

df['notice_period_score'] = (
    df['notice_period_days']
      .apply(calculate_notice_score)
)

# =====================================================
# PREFERRED WORK MODE SCORE
#
# remote= -100 hybrid = flexible = onsite = 100% because work mode is Hybrid.
# =====================================================

work_mode_scores = {
    'remote': -100,
    'hybrid': 100,
    'flexible': 100,
    'onsite': 100
}

df['preferred_work_mode_score'] = (
    df['preferred_work_mode']
      .astype(str)
      .str.strip()
      .str.lower()
      .map(work_mode_scores)
      .fillna(0)
)

# =====================================================
# OFFER ACCEPTANCE RATE SCORE
# =====================================================

df['offer_acceptance_rate'] = pd.to_numeric(
    df['offer_acceptance_rate'],
    errors='coerce'
)

df['offer_acceptance_score'] = (
    (df['offer_acceptance_rate']*100)
).round(2)

df['final_candidate_signal_score'] = (
      0.30 * df['last_active_score']
    + 0.50 * df['notice_period_score']
    + 0.15 * df['preferred_work_mode_score']
    + 0.025 * df['open_to_work_score']
    + 0.025 * df['offer_acceptance_score']
)

# =====================================================
# OUTPUT FILE
# =====================================================

output_df = df[[
    'candidate_id',
    'last_active_score',
    'open_to_work_score',
    'notice_period_score',
    'preferred_work_mode_score',
    'offer_acceptance_score',
    'final_candidate_signal_score'
]]


OUTPUT_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_redrob_signals_scored.csv"

output_df.to_csv(
    OUTPUT_FILE,
    index=False
)

# =====================================================
# SUMMARY
# =====================================================

print("Output Shape:", output_df.shape)

print("\nSample Results:")
print(output_df.head())

print(f"\nScored file saved as: {OUTPUT_FILE}")

Output Shape: (21673, 7)

Sample Results:
   candidate_id  last_active_score  open_to_work_score  notice_period_score  \
0  CAND_0000005             -47.40                 100                  100   
1  CAND_0000007              81.92                -100                  100   
2  CAND_0000021             -19.45                -100                  100   
3  CAND_0000024              13.42                -100                  100   
4  CAND_0000031              81.37                 100                  100   

   preferred_work_mode_score  offer_acceptance_score  \
0                        100                  -100.0   
1                        100                  -100.0   
2                        100                  -100.0   
3                        100                  -100.0   
4                        100                    38.0   

   final_candidate_signal_score  
0                        50.780  
1                        84.576  
2                        54.165  
3         

Creating final score from semantic score and redrob signal score

In [ ]:
signal_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_redrob_signals_scored.csv")
semantic_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores.csv")

# Keep required columns from signal file
signal_df = signal_df[
    ['candidate_id', 'final_candidate_signal_score']
]

# Merge on candidate_id
final_df = semantic_df.merge(
    signal_df,
    on='candidate_id',
    how='left'
)

# Normalize signal score (-100 to 100 -> -1 to 1)
final_df['signal_score_normalized'] = (
    final_df['final_candidate_signal_score'] / 100
)

# Final Score cosidered only 20% from redrob signal score becuase two things are important from that file rest should be considered from sematic score
final_df['final_score'] = (
      0.80 * final_df['semantic_score']
    + 0.20 * final_df['signal_score_normalized']
).round(4)

# Sort by final score descending
final_df = final_df.sort_values(
    by='final_score',
    ascending=False
)

# Keep required columns
final_output = final_df[
    [
        'candidate_id',
        'semantic_score',
        'final_candidate_signal_score',
        'final_score'
    ]
]

# Save
final_output.to_csv(
    "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_final_scores.csv",
    index=False
)

print(final_output.head(10))

    candidate_id  semantic_score  final_candidate_signal_score  final_score
0   CAND_0098454        0.832732                        89.903       0.8460
12  CAND_0043860        0.817613                        94.241       0.8426
37  CAND_0048558        0.810641                        93.553       0.8356
8   CAND_0068932        0.818909                        89.741       0.8346
21  CAND_0032955        0.815166                        91.162       0.8345
13  CAND_0069638        0.817339                        88.576       0.8310
7   CAND_0082618        0.819378                        87.352       0.8302
16  CAND_0073418        0.816740                        88.222       0.8298
68  CAND_0056617        0.807098                        91.899       0.8295
64  CAND_0091890        0.807291                        91.567       0.8290


Creating final file

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_final_scores.csv")

# Take first 100 rows
top_100 = df.head(100).copy()

# Create rank column
top_100["rank"] = range(1, len(top_100) + 1)

# Select required columns and rename final_score to score
result_df = top_100[["candidate_id", "rank", "final_score"]].rename(
    columns={"final_score": "score"}
)


# Save to a new CSV file
result_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/top_100_candidates.csv", index=False)

print("Top 100 candidates with there score against job description is saved...")

Top 100 candidates with there score against job description is saved...


Creating the reason for top 100 candidates.

filter the profile file, skill, career history and redrob signal files.



In [ ]:
top_candidates = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/top_100_candidates.csv")

# Profile file to filter
candidateProfile_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_profile.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateProfile_df[
    candidateProfile_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_candidateprofile_file.csv", index=False)

# Career history file to filter
candidateCareerHistory_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_career_history.csv")

filtered_df = candidateCareerHistory_df[
    candidateCareerHistory_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_candidatecareerhistory_file.csv", index=False)

#redrob signal file to filter
candidateredrobsignal_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_redrob_signals.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateredrobsignal_df[
    candidateredrobsignal_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_redrobsignal_file.csv", index=False)

#skills file to filter
candidateskills_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_skills.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateskills_df[
    candidateskills_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_skills_file.csv", index=False)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# =====================================================
# CONFIG
# =====================================================

TOP100_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/top_100_candidates.csv"
PROFILE_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_candidateprofile_file.csv"
CAREER_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_candidatecareerhistory_file.csv"
SIGNAL_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_redrobsignal_file.csv"
SKILLS_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_skills_file.csv"
OUTPUT_FILE= "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidates_submission.csv"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:


print("Loading files...")

top100 = pd.read_csv(TOP100_FILE)
profile = pd.read_csv(PROFILE_FILE)
career = pd.read_csv(CAREER_FILE)
signals = pd.read_csv(SIGNAL_FILE)
skills = pd.read_csv(SKILLS_FILE)

with open(JSON_FILE, "r") as f:
    jd = json.load(f)

print("Files loaded.")

# ==========================================================
# SMALL JD SUMMARY
# ==========================================================

jd_summary = {
    "experience": jd["experience"],
    "preferred_roles": jd["roles"]["preferred_titles"],
    "required_skills": [
        s["skill_id"] for s in jd["skills"]["core_required"]
    ],
    "domains": [
        d["domain_id"]
        for d in jd["experience_requirements"]["required_domains"]
    ]
}

# ==========================================================
# LOAD MODEL
# ==========================================================


print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded.")

# ==========================================================
# GENERATE REASON
# ==========================================================

def generate_reason(candidate_id):

    profile_df = profile[
        profile["candidate_id"] == candidate_id
    ]

    career_df = career[
        career["candidate_id"] == candidate_id
    ]

    signal_df = signals[
        signals["candidate_id"] == candidate_id
    ]

    skills_df = skills[
        skills["candidate_id"] == candidate_id
    ]

    # ----------------------------
    # Candidate Summary
    # ----------------------------

    current_title = ""
    years_exp = ""

    if not profile_df.empty:

        if "current_title" in profile_df.columns:
            current_title = profile_df.iloc[0]["current_title"]

        if "years_of_exp" in profile_df.columns:
            years_exp = profile_df.iloc[0]["years_of_exp"]

    industries = []

    if (
        not career_df.empty
        and "industries" in career_df.columns
    ):
        industries = career_df["industries"].dropna().unique().tolist()

    offer_rate = ""

    if (
        not signal_df.empty
        and "offer_acceptance_rate" in signal_df.columns
    ):
        offer_rate = signal_df.iloc[0]["offer_acceptance_rate"]

    candidate_skills = []

    # Change "skill_name" if your column name is different
    if (
        not skills_df.empty
        and "skills" in skills_df.columns
    ):
        candidate_skills = (
            skills_df["skills"]
            .dropna()
            .unique()
            .tolist()
        )

    candidate_summary = {
        "current_title": current_title,
        "years_of_experience": years_exp,
        "industries": industries,
        "offer_acceptance_rate": offer_rate,
        "skills": candidate_skills
    }

    # ----------------------------
    # Prompt
    # ----------------------------

    prompt = f"""
You are an AI recruiter.

Write EXACTLY ONE sentence.

Maximum 15 words.

Mention only the strongest match between the candidate and the job.

Job:
{jd_summary}

Candidate:
{candidate_summary}

Return ONLY the sentence.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert recruiter."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            temperature=0.0,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    reason = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    if "." in reason:
        reason = reason.split(".")[0] + "."

    return reason


# ==========================================================
# GENERATE FOR TOP 100
# ==========================================================

reasons = []

total = len(top100)

for i, row in top100.iterrows():

    candidate_id = row["candidate_id"]

    print(f"{i+1}/{total} : {candidate_id}")

    try:
        reason = generate_reason(candidate_id)
    except Exception as e:
        print(e)
        reason = ""

    reasons.append(reason)

# ==========================================================
# SAVE
# ==========================================================

top100["reason"] = reasons

top100.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nCompleted Successfully.")
print("Saved:", OUTPUT_FILE)

Loading files...
Files loaded.
Loading model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded.
1/100 : CAND_0098454
2/100 : CAND_0043860
3/100 : CAND_0048558
4/100 : CAND_0068932
5/100 : CAND_0032955
6/100 : CAND_0069638
7/100 : CAND_0082618
8/100 : CAND_0073418
9/100 : CAND_0056617
10/100 : CAND_0091890
11/100 : CAND_0081932
12/100 : CAND_0076107
13/100 : CAND_0016657
14/100 : CAND_0005477
15/100 : CAND_0070333
16/100 : CAND_0003575
17/100 : CAND_0068513
18/100 : CAND_0045475
19/100 : CAND_0029272
20/100 : CAND_0094728
21/100 : CAND_0018888
22/100 : CAND_0012840
23/100 : CAND_0099347
24/100 : CAND_0001819
25/100 : CAND_0028426
26/100 : CAND_0025335
27/100 : CAND_0044696
28/100 : CAND_0025882
29/100 : CAND_0082407
30/100 : CAND_0057162
31/100 : CAND_0019845
32/100 : CAND_0032216
33/100 : CAND_0091580
34/100 : CAND_0066951
35/100 : CAND_0025923
36/100 : CAND_0007631
37/100 : CAND_0084161
38/100 : CAND_0042056
39/100 : CAND_0072688
40/100 : CAND_0052431
41/100 : CAND_0065430
42/100 : CAND_0084681
43/100 : CAND_0021442
44/100 : CAND_0000273
45/100 : CAND_0091371
46/10

Try with vector DB created with BEG embedding model

In [10]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"
# =====================================================
# LOAD JOB REQUIREMENTS JSON
# =====================================================

with open(JSON_FILE, "r") as f:
    jd = json.load(f)

# =====================================================
# BUILD SEMANTIC QUERY
# =====================================================

core_skills = [
    s["skill_id"]
    for s in jd["skills"]["core_required"]
]

secondary_skills = [
    s["skill_id"]
    for s in jd["skills"]["secondary_skills"]
]

domains = [
    d["domain_id"]
    for d in jd["experience_requirements"]["required_domains"]
]

roles = jd["roles"]["preferred_titles"]

industries = jd["company"]["industry_constraints"]["preferred_industries"]

cities = jd["location"]["domestic"]["preferred_cities"]

jd_text = f"""
Hiring for an AI/ML Search Engineer.

Preferred Roles:
{', '.join(roles)}

Mandatory Skills:
{', '.join(core_skills)}

Preferred Skills:
{', '.join(secondary_skills)}

Required Domains:
{', '.join(domains)}

Preferred Industries:
{', '.join(industries)}

Preferred Locations:
{', '.join(cities)}

Experience:
{jd['experience']['min_years']} to {jd['experience']['max_years']} years

Engineering Style:
{jd['work_style']['engineering_style']}

Communication Model:
{jd['work_style']['communication_model']}
"""

print("=" * 100)
print("JD QUERY")
print("=" * 100)
print(jd_text)

# =====================================================
# LOAD EMBEDDING MODEL
# =====================================================

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE JD EMBEDDING
# =====================================================

jd_embedding = model.encode(
    [jd_text],
    normalize_embeddings=True
).astype(np.float32)

JD QUERY

Hiring for an AI/ML Search Engineer.

Preferred Roles:
SENIOR_AI_ENGINEER, ML_ENGINEER, SEARCH_ENGINEER, RANKING_ENGINEER, RECOMMENDER_SYSTEMS_ENGINEER, APPLIED_SCIENTIST

Mandatory Skills:
EMBEDDINGS, RETRIEVAL_SYSTEMS, VECTOR_DB, HYBRID_SEARCH, PYTHON, RANKING_SYSTEMS_EVALUATION, EVALUATION_FRAMEWORKS, AB_TESTING

Preferred Skills:
LLM_FINE_TUNING, LEARNING_TO_RANK, HR_TECH, DISTRIBUTED_SYSTEMS, OPEN_SOURCE_CONTRIBUTION

Required Domains:
RETRIEVAL_SYSTEMS, RANKING_SYSTEMS, SEARCH_SYSTEMS, ML_PRODUCTION_SYSTEMS

Preferred Industries:
AI_NATIVE_TALENT_INTELLIGENCE, RECRUITMENT_TECH, SEARCH_INFRASTRUCTURE, RECOMMENDATION_SYSTEMS, HR_TECH, APPLIED_ML

Preferred Locations:
PUNE, NOIDA

Experience:
5 to 9 years

Engineering Style:
FULL_STACK_ML_SYSTEMS

Communication Model:
ASYNC_WRITING_HEAVY



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
FAISS_INDEX_FILE = '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/database/BGE/candSummaryvector_bgedb.faiss'
METADATA_FILE = '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/database/BGE/candSummarymetadata_bge.pkl'
# =====================================================
# LOAD FAISS
# =====================================================

index = faiss.read_index(FAISS_INDEX_FILE)

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

print("Candidates:", len(metadata))

# =====================================================
# SEARCH ALL CANDIDATES
# =====================================================

k = index.ntotal

scores, indices = index.search(
    jd_embedding,
    k
)

Candidates: 21673


In [12]:
# =====================================================
# BUILD RESULT CSV
# =====================================================

results = []

for rank, idx in enumerate(indices[0]):

    similarity = float(scores[0][rank])

    # cosine similarity -> score 0-1
    semantic_score = (similarity + 1) / 2


    candidate = metadata[idx]

    results.append({
        "candidate_id": candidate.get("candidate_id"),
        "semantic_similarity": round(similarity, 4),
        "semantic_score": semantic_score
    })

# =====================================================
# SAVE OUTPUT
# =====================================================

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    "semantic_score",
    ascending=False
)

OUTPUT_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores_bge.csv"

result_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")
print(result_df.head(20))

Saved: /content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores_bge.csv
    candidate_id  semantic_similarity  semantic_score
0   CAND_0094759               0.8455        0.922748
1   CAND_0018499               0.8385        0.919242
2   CAND_0046525               0.8365        0.918227
3   CAND_0022812               0.8361        0.918045
4   CAND_0051004               0.8356        0.917816
5   CAND_0081846               0.8341        0.917041
6   CAND_0005649               0.8284        0.914200
7   CAND_0039754               0.8280        0.913984
8   CAND_0057563               0.8248        0.912401
9   CAND_0061339               0.8237        0.911863
10  CAND_0086151               0.8230        0.911493
11  CAND_0010257               0.8229        0.911435
12  CAND_0005260               0.8225        0.911234
13  CAND_0031593               0.8217        0.910827
14  CAND_0091534               0.8215        0.910743
15  CAND_0074225        

In [13]:
signal_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_redrob_signals_scored.csv")
semantic_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores_bge.csv")

# Keep required columns from signal file
signal_df = signal_df[
    ['candidate_id', 'final_candidate_signal_score']
]

# Merge on candidate_id
final_df = semantic_df.merge(
    signal_df,
    on='candidate_id',
    how='left'
)

# Normalize signal score (-100 to 100 -> -1 to 1)
final_df['signal_score_normalized'] = (
    final_df['final_candidate_signal_score'] / 100
)

# Final Score cosidered only 20% from redrob signal score becuase two things are important from that file rest should be considered from sematic score
final_df['final_score'] = (
      0.80 * final_df['semantic_score']
    + 0.20 * final_df['signal_score_normalized']
).round(4)

# Sort by final score descending
final_df = final_df.sort_values(
    by='final_score',
    ascending=False
)

# Keep required columns
final_output = final_df[
    [
        'candidate_id',
        'semantic_score',
        'final_candidate_signal_score',
        'final_score'
    ]
]

# Save
final_output.to_csv(
    "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_final_scores_bge.csv",
    index=False
)

print(final_output.head(10))

     candidate_id  semantic_score  final_candidate_signal_score  final_score
2    CAND_0046525        0.918227                        92.946       0.9205
4    CAND_0051004        0.917816                        90.188       0.9146
7    CAND_0039754        0.913984                        91.642       0.9145
21   CAND_0068811        0.909173                        93.244       0.9138
36   CAND_0000031        0.907407                        92.861       0.9116
106  CAND_0043860        0.900428                        94.241       0.9088
64   CAND_0081852        0.903085                        92.771       0.9080
115  CAND_0048558        0.899964                        93.553       0.9071
57   CAND_0032955        0.903615                        91.162       0.9052
39   CAND_0018549        0.906687                        89.861       0.9051


In [14]:
df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_final_scores_bge.csv")

# Take first 100 rows
top_100 = df.head(100).copy()

# Create rank column
top_100["rank"] = range(1, len(top_100) + 1)

# Select required columns and rename final_score to score
result_df = top_100[["candidate_id", "rank", "final_score"]].rename(
    columns={"final_score": "score"}
)


# Save to a new CSV file
result_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/top_100_candidates_bge.csv", index=False)

print("Top 100 candidates with there score against job description is saved...")

Top 100 candidates with there score against job description is saved...


In [15]:
top_candidates = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/top_100_candidates_bge.csv")

# Profile file to filter
candidateProfile_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_profile.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateProfile_df[
    candidateProfile_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_candidateprofile_file_bge.csv", index=False)

# Career history file to filter
candidateCareerHistory_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_career_history.csv")

filtered_df = candidateCareerHistory_df[
    candidateCareerHistory_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_candidatecareerhistory_file_bge.csv", index=False)

#redrob signal file to filter
candidateredrobsignal_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_redrob_signals.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateredrobsignal_df[
    candidateredrobsignal_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_redrobsignal_file_bge.csv", index=False)

#skills file to filter
candidateskills_df = pd.read_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_skills.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateskills_df[
    candidateskills_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_skills_file_bge.csv", index=False)


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [17]:
# =====================================================
# CONFIG
# =====================================================

TOP100_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/top_100_candidates_bge.csv"
PROFILE_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_candidateprofile_file_bge.csv"
CAREER_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_candidatecareerhistory_file_bge.csv"
SIGNAL_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_redrobsignal_file_bge.csv"
SKILLS_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/top100candidatesfiltereddataset/filtered_skills_file_bge.csv"
OUTPUT_FILE= "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidates_submission_bge.csv"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [18]:
print("Loading files...")

top100 = pd.read_csv(TOP100_FILE)
profile = pd.read_csv(PROFILE_FILE)
career = pd.read_csv(CAREER_FILE)
signals = pd.read_csv(SIGNAL_FILE)
skills = pd.read_csv(SKILLS_FILE)

with open(JSON_FILE, "r") as f:
    jd = json.load(f)

print("Files loaded.")

# ==========================================================
# SMALL JD SUMMARY
# ==========================================================

jd_summary = {
    "experience": jd["experience"],
    "preferred_roles": jd["roles"]["preferred_titles"],
    "required_skills": [
        s["skill_id"] for s in jd["skills"]["core_required"]
    ],
    "domains": [
        d["domain_id"]
        for d in jd["experience_requirements"]["required_domains"]
    ]
}

# ==========================================================
# LOAD MODEL
# ==========================================================


print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded.")

# ==========================================================
# GENERATE REASON
# ==========================================================

def generate_reason(candidate_id):

    profile_df = profile[
        profile["candidate_id"] == candidate_id
    ]

    career_df = career[
        career["candidate_id"] == candidate_id
    ]

    signal_df = signals[
        signals["candidate_id"] == candidate_id
    ]

    skills_df = skills[
        skills["candidate_id"] == candidate_id
    ]

    # ----------------------------
    # Candidate Summary
    # ----------------------------

    current_title = ""
    years_exp = ""

    if not profile_df.empty:

        if "current_title" in profile_df.columns:
            current_title = profile_df.iloc[0]["current_title"]

        if "years_of_exp" in profile_df.columns:
            years_exp = profile_df.iloc[0]["years_of_exp"]

    industries = []

    if (
        not career_df.empty
        and "industries" in career_df.columns
    ):
        industries = career_df["industries"].dropna().unique().tolist()

    offer_rate = ""

    if (
        not signal_df.empty
        and "offer_acceptance_rate" in signal_df.columns
    ):
        offer_rate = signal_df.iloc[0]["offer_acceptance_rate"]

    candidate_skills = []

    # Change "skill_name" if your column name is different
    if (
        not skills_df.empty
        and "skills" in skills_df.columns
    ):
        candidate_skills = (
            skills_df["skills"]
            .dropna()
            .unique()
            .tolist()
        )

    candidate_summary = {
        "current_title": current_title,
        "years_of_experience": years_exp,
        "industries": industries,
        "offer_acceptance_rate": offer_rate,
        "skills": candidate_skills
    }

    # ----------------------------
    # Prompt
    # ----------------------------

    prompt = f"""
You are an AI recruiter.

Write EXACTLY ONE sentence.

Maximum 15 words.

Mention only the strongest match between the candidate and the job.

Job:
{jd_summary}

Candidate:
{candidate_summary}

Return ONLY the sentence.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert recruiter."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            temperature=0.0,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    reason = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    if "." in reason:
        reason = reason.split(".")[0] + "."

    return reason


# ==========================================================
# GENERATE FOR TOP 100
# ==========================================================

reasons = []

total = len(top100)

for i, row in top100.iterrows():

    candidate_id = row["candidate_id"]

    print(f"{i+1}/{total} : {candidate_id}")

    try:
        reason = generate_reason(candidate_id)
    except Exception as e:
        print(e)
        reason = ""

    reasons.append(reason)

# ==========================================================
# SAVE
# ==========================================================

top100["reason"] = reasons

top100.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nCompleted Successfully.")
print("Saved:", OUTPUT_FILE)

Loading files...
Files loaded.
Loading model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded.
1/100 : CAND_0046525
2/100 : CAND_0051004
3/100 : CAND_0039754
4/100 : CAND_0068811
5/100 : CAND_0000031
6/100 : CAND_0043860
7/100 : CAND_0081852
8/100 : CAND_0048558
9/100 : CAND_0032955
10/100 : CAND_0018549
11/100 : CAND_0017960
12/100 : CAND_0091890
13/100 : CAND_0068410
14/100 : CAND_0003575
15/100 : CAND_0070514
16/100 : CAND_0005260
17/100 : CAND_0010770
18/100 : CAND_0005477
19/100 : CAND_0056617
20/100 : CAND_0073418
21/100 : CAND_0075481
22/100 : CAND_0076251
23/100 : CAND_0099806
24/100 : CAND_0070333
25/100 : CAND_0082618
26/100 : CAND_0045475
27/100 : CAND_0015967
28/100 : CAND_0025882
29/100 : CAND_0099347
30/100 : CAND_0086022
31/100 : CAND_0087841
32/100 : CAND_0068513
33/100 : CAND_0076107
34/100 : CAND_0026716
35/100 : CAND_0062247
36/100 : CAND_0012840
37/100 : CAND_0000273
38/100 : CAND_0091534
39/100 : CAND_0034177
40/100 : CAND_0001819
41/100 : CAND_0050454
42/100 : CAND_0032216
43/100 : CAND_0033179
44/100 : CAND_0098454
45/100 : CAND_0043384
46/10